# 05 — Kiran et al. (2025) HQCAN: 4-qubit variational quantum classifier vs classical models on the same 4 PCs

**Method (ICICI):** PCA to 4 components → AngleEmbedding(RY) → 2 StronglyEntanglingLayers →
⟨Z₀⟩ → logistic output; 80/20 split on Pima (claimed 99.7 % accuracy). Here the same circuit runs
on all 4 diabetes datasets, next to logistic regression and an RBF-SVM **on the same 4 PCs**, so any
gap is due to the model, not the features.

**Protocols**
* `paper` — scaling + PCA fitted on all rows, 80/20 stratified split (seed 42), ungrouped.
* `clean` — scaling + PCA in-fold; grouped/stratified 5-fold × seeds {0, 1, 42}.

Training is fixed in advance (no tuning on test folds): Adam, lr 0.05, 60 full-batch epochs,
class-balanced BCE loss, weights seeded per run. Needs `pennylane` + `torch` (cell below installs
PennyLane if missing; Colab/Kaggle already have torch). Runtime ≈ 10–25 min on CPU.

In [1]:
import os, sys, subprocess
from pathlib import Path
for p in [Path.cwd(), Path.cwd() / "notebooks2", Path("/content/notebooks2"), *Path("/kaggle/input").glob("*"), *Path("/kaggle/input").glob("*/notebooks2")]:
    if (p / "nb2_common.py").exists():
        sys.path.insert(0, str(p)); break
try:
    import pennylane as qml
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pennylane"]); import pennylane as qml
import torch
from nb2_common import *
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
sanity_check_kernels()
DATA = load_all()
print("pennylane", qml.__version__, "torch", torch.__version__)

nb2_common loaded: numpy 2.4.6, sklearn 1.6.1
kernel self-test passed
pima     N=  768 d=  8 prevalence=0.349 unique=  768 grouped_cv=False  [diabetes]
sylhet   N=  520 d= 16 prevalence=0.615 unique=  251 grouped_cv=True  [diabetes (early stage)]


cdc      N= 2000 d= 21 prevalence=0.140 unique= 1995 grouped_cv=True  [diabetes or prediabetes (self-reported)]


hosp130  N= 2000 d= 44 prevalence=0.088 unique= 2000 grouped_cv=False  [readmitted within 30 days]
pennylane 0.45.1 torch 2.14.1+cu130


In [2]:
NQ, LAYERS, EPOCHS, LR = 4, 2, 60, 0.05
dev = qml.device("default.qubit", wires=NQ)


@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit(x, w):
    qml.AngleEmbedding(x, wires=range(NQ), rotation="Y")
    qml.StronglyEntanglingLayers(w, wires=range(NQ))
    return qml.expval(qml.PauliZ(0))


def train_vqc(Xtr, ytr, Xte, seed):
    torch.manual_seed(seed)
    shape = qml.StronglyEntanglingLayers.shape(n_layers=LAYERS, n_wires=NQ)
    w = torch.nn.Parameter(0.1 * torch.randn(shape, dtype=torch.float64))
    a = torch.nn.Parameter(torch.tensor([2.0, 0.0], dtype=torch.float64))  # logit = a0*<Z> + a1
    xt = torch.tensor(Xtr, dtype=torch.float64); yt = torch.tensor(ytr, dtype=torch.float64)
    pos = max(ytr.mean(), 1e-3)
    pw = torch.tensor((1 - pos) / pos, dtype=torch.float64)
    opt = torch.optim.Adam([w, a], lr=LR)
    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=pw)
    for _ in range(EPOCHS):
        opt.zero_grad()
        loss = lossf(a[0] * circuit(xt, w) + a[1], yt)
        loss.backward(); opt.step()
    with torch.no_grad():
        p = torch.sigmoid(a[0] * circuit(torch.tensor(Xte, dtype=torch.float64), w) + a[1]).numpy()
    return (p >= 0.5).astype(int), p


def pcs(X_fit, *others, seed=0):
    """standardise -> PCA(4) -> MinMax [0, pi], all fitted on X_fit."""
    return to_qubits(X_fit, *others, q=NQ, seed=seed)


def kiran_method(Xtr, ytr, Xte, seed, groups_tr, X_prep):
    Qr, Qe = prep(pcs, X_prep, Xtr, Xte, seed=seed)
    out = {"HQCAN VQC (4q, 2 layers)": train_vqc(Qr, ytr, Qe, seed)}
    lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Qr, ytr)
    out["Logistic Regression (same 4 PCs, balanced)"] = (lr.predict(Qe), lr.predict_proba(Qe)[:, 1])
    svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=seed).fit(Qr, ytr)
    out["RBF-SVM (same 4 PCs, balanced)"] = (svm.predict(Qe), svm.decision_function(Qe))
    out["Angle-QSVM (same 4 PCs)"] = svm_precomputed(angle_kernel(Qr, Qr), ytr, angle_kernel(Qe, Qr), seed=seed)
    return out


rec = Recorder("P05_Kiran", "05_kiran", config=dict(
    method=f"VQC: PCA{NQ} -> AngleEmbedding(Y) -> StronglyEntanglingLayers x{LAYERS} -> <Z0>; Adam lr {LR}, {EPOCHS} epochs, balanced BCE",
    paper_protocol="global scaling+PCA, 80/20 stratified split seed 42",
    pennylane=qml.__version__, torch=torch.__version__))
print("paper protocol"); run_paper_holdout(rec, DATA, kiran_method, test_size=0.2, global_prep=True)
print("clean protocol"); run_clean(rec, DATA, kiran_method)
df = rec.save()

paper protocol


  paper  pima     test rows sharing a feature vector with train: 0


  paper  sylhet   test rows sharing a feature vector with train: 72


  paper  cdc      test rows sharing a feature vector with train: 1


  paper  hosp130  test rows sharing a feature vector with train: 0
clean protocol


  clean  pima     done in   23.1s


  clean  sylhet   done in   20.8s


  clean  cdc      done in   36.0s


  clean  hosp130  done in   34.2s
saved /content/notebooks2/results/05_kiran.csv  (256 rows)


In [3]:
pd.set_option("display.width", 220, "display.max_rows", 200)
summarise(df)

,dataset,protocol,model,accuracy,balanced_accuracy,recall,specificity,roc_auc,n_runs
0,cdc,clean,Angle-QSVM (same 4 PCs),0.86 ± 0.018,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.0,0.708 ± 0.031,15
1,cdc,clean,"HQCAN VQC (4q, 2 layers)",0.716 ± 0.018,0.715 ± 0.026,0.714 ± 0.048,0.716 ± 0.02,0.786 ± 0.019,15
2,cdc,clean,"Logistic Regression (same 4 PCs, balanced)",0.723 ± 0.016,0.721 ± 0.029,0.72 ± 0.064,0.722 ± 0.021,0.8 ± 0.016,15
3,cdc,clean,"RBF-SVM (same 4 PCs, balanced)",0.686 ± 0.017,0.727 ± 0.026,0.786 ± 0.061,0.669 ± 0.026,0.798 ± 0.015,15
4,cdc,paper,Angle-QSVM (same 4 PCs),0.86 ± nan,0.5 ± nan,0.0 ± nan,1.0 ± nan,0.718 ± nan,1
5,cdc,paper,"HQCAN VQC (4q, 2 layers)",0.732 ± nan,0.725 ± nan,0.714 ± nan,0.735 ± nan,0.812 ± nan,1
6,cdc,paper,"Logistic Regression (same 4 PCs, balanced)",0.735 ± nan,0.726 ± nan,0.714 ± nan,0.738 ± nan,0.8 ± nan,1
7,cdc,paper,"RBF-SVM (same 4 PCs, balanced)",0.705 ± nan,0.724 ± nan,0.75 ± nan,0.698 ± nan,0.792 ± nan,1
8,hosp130,clean,Angle-QSVM (same 4 PCs),0.912 ± 0.001,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.0,0.52 ± 0.056,15
9,hosp130,clean,"HQCAN VQC (4q, 2 layers)",0.569 ± 0.048,0.536 ± 0.046,0.497 ± 0.084,0.576 ± 0.053,0.553 ± 0.06,15


## Is the VQC better than LR on the same 4 PCs? Paired per-fold difference (clean protocol)

In [4]:
c = df[df.protocol == "clean"].pivot_table(index=["dataset", "seed", "fold"], columns="model", values="balanced_accuracy")
diff = (c["HQCAN VQC (4q, 2 layers)"] - c["Logistic Regression (same 4 PCs, balanced)"]).groupby("dataset")
diff.agg(["mean", "std", "min", "max"]).round(3)

,mean,std,min,max
dataset,,,,
cdc,-0.006,0.029,-0.044,0.044
hosp130,-0.006,0.042,-0.071,0.076
pima,-0.017,0.025,-0.063,0.014
sylhet,-0.138,0.120,-0.443,-0.009
